In [0]:
%run /Configurations/Init_Scripts

In [0]:

dbutils.widgets.text("CatalogName", "cd_prod")
CatalogName = dbutils.widgets.get("CatalogName")


dbutils.widgets.text("ExternalLocationName", "mntprod")
ExternalLocationName = dbutils.widgets.get("ExternalLocationName")
filePrefix = '/dbfs/'+ExternalLocationName

dbutils.widgets.text('ExternalLocation_raw',"/mntprod_raw")
ExternalLocation_raw = dbutils.widgets.get('ExternalLocation_raw')

dbutils.widgets.text("ExternalLocationName_silver", "/mntprod_silver")
ExternalLocationName_silver = dbutils.widgets.get("ExternalLocationName_silver")


In [0]:
spark.conf.set("spark.sql.catalog.current", CatalogName)

In [0]:
# %sql
# drop table if exists promotion.DIM_RuleEngineConfig;
# CREATE TABLE IF NOT EXISTS promotion.DIM_RuleEngineConfig (
# RuleID BIGINT GENERATED ALWAYS AS IDENTITY, 
# RuleName STRING,
# RuleCategory STRING,
# RuleCode STRING,
# RuleComments STRING,
# RuleOrder int,
# PromotionUUID STRING,
# DependentRules STRING,
# EffectiveDateTime Date,
# TerminationDateTime Date,
# CreatedDate Date,
# CreatedBy String,
# UpdatedDate Date,
# UpdatedBy String,
# Active boolean
# )
# USING delta
# LOCATION 'dbfs:/mnt/silver/DIMRuleEngineConfig';



In [0]:
%sql
drop table if exists promotion.STG_RuleEngineConfig;
CREATE TABLE promotion.STG_RuleEngineConfig AS SELECT * FROM promotion.DIM_RuleEngineConfig WHERE 1=0;

In [0]:
# %sql
# delete from promotion.DIM_RuleEngineConfig;


In [0]:
PromotionName = '4P Beta'
PromotionUUID_US = (spark.read.format('delta').table('promotion.DIM_Promotion'))\
    .select("PromotionUUID")\
    .filter(col("PromotionName") == f"{PromotionName}" )\
    .head()[0]

In [0]:
SubscriptionDateChangeFlag = '''CASE WHEN upper(ExceptionSubType) = "SUBSCRIPTION DATE CHANGE" THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('SubscriptionDateChangeFlag', 'InvoiceException', '{}','Check exception qualification PerpatientFee-NoPatientFee and   PreInvoice-PostInvoice and populate respective comments', 1,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(SubscriptionDateChangeFlag,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)

Comments = '''CASE WHEN SubscriptionDateChangeFlag = 1 THEN "Exception_SubscriptionDateChange" WHEN IsInvoiced = 0 AND ExceptionCreditAmount = 950 THEN "Exception_PerPatientPreInvoice" WHEN IsInvoiced = 1 AND ExceptionCreditAmount = 950 THEN "Exception_PerPatientPostInvoice" WHEN ExceptionCreditAmount = 0 THEN "Exception_PerPatientPostInvoice" WHEN IsInvoiced = 0 AND ExceptionCreditAmount = CycleCount * 200 THEN "Exception_PerCycleFeePreInvoice" WHEN IsInvoiced = 1 AND ExceptionCreditAmount = CycleCount * 200 THEN "Exception_PerCycleFeePostInvoice" WHEN IsInvoiced = 0 AND ExceptionCreditAmount <> 950 THEN "Exception_NoPatientPreInvoice" WHEN IsInvoiced = 1 AND ExceptionCreditAmount <> 950 THEN "Exception_NoPatientPostInvoice" END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('Comments', 'InvoiceException', '{}','Check exception qualification PerpatientFee-NoPatientFee and   PreInvoice-PostInvoice and populate respective comments', 2,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(Comments,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)

NewSubscriptionEndDate = '''CASE WHEN SubscriptionDateChangeFlag = 1 THEN ConsumerSubscriptionEndDate WHEN SubscriptionEndDate > InvoiceException_CreatedDate THEN InvoiceException_CreatedDate ELSE SubscriptionEndDate END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('NewSubscriptionEndDate', 'InvoiceException', '{}','Calculate subscriptionEndDate based on exception date', 3,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(NewSubscriptionEndDate,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)


PatientClassificationName = '''CASE WHEN SubscriptionDateChangeFlag = 1 AND ExceptionCreditAmount = 950 THEN "PerPatientFee" WHEN IsInvoiced = 0 AND ExceptionCreditAmount = 950 THEN "PerPatientFeeExceptionPreInvoice" WHEN IsInvoiced = 1 AND ExceptionCreditAmount = 950 THEN "PerPatientFeeExceptionPostInvoice" WHEN IsInvoiced = 0 AND ExceptionCreditAmount = CycleCount * 200 THEN "PerCycleFeeExceptionPreInvoice" WHEN IsInvoiced = 1 AND ExceptionCreditAmount = CycleCount * 200 THEN "PerCycleFeeExceptionPostInvoice" WHEN IsInvoiced = 0 AND ExceptionCreditAmount <> 950 THEN "NoPatientAssociationFeeExceptionPreInvoice" WHEN IsInvoiced = 1 AND ExceptionCreditAmount <> 950 THEN "NoPatientAssociationFeeExceptionPostInvoice" END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('PatientClassificationName', 'InvoiceException', '{}','Check exception qualification PerpatientFee-NoPatientFee and PreInvoice-PostInvoice and populate respective classifaction', 4,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(PatientClassificationName,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)

CoolSculptingID = '''CASE WHEN coalesce(CAST(CoolSculptingID AS BIGINT),0) = 0 THEN "MISSING" ELSE CoolSculptingID END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('CoolSculptingID', 'CycleFlag', '{}','Transform CoolScultingID to MISSING for all invalid values', 1,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(CoolSculptingID,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)

NewSubscriptionFlag = '''CASE WHEN ConsumerSubscriptionUUID IS NULL THEN 1 ELSE 0 END '''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('NewSubscriptionFlag', 'CycleFlag', '{}','Rule to check New Subscription for Live Data', 2,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(NewSubscriptionFlag,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)

AdjustSubscriptionFlag = '''CASE WHEN PilotSubscriptionFlag = "Y" THEN 0 WHEN CycleDate > PreviousSubscriptionEndDate AND CycleDate < SubscriptionStartDate THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('AdjustSubscriptionFlag', 'CycleFlag', '{}','Rule to check Late Cycle which needs to adjust consumer subscription', 3,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(AdjustSubscriptionFlag,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)

NoPhoneNumberFlag = '''CASE WHEN CoolSculptingID IS NULL OR CoolSculptingID IN ("UNKNOWN","MISSING","","0000000000") THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('NoPhoneNumberFlag', 'CycleFlag', '{}','Rule to check for CoolsculptingID and determins if NoPhoneNumberFlag', 4,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(NoPhoneNumberFlag,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)

LateCycleFlag = '''CASE WHEN CycleDate <= MaxInvoiceDate THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('LateCycleFlag', 'CycleFlag', '{}','Rule to check Late Cycle which needs to adjust consumer subscription', 5,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(LateCycleFlag,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)

FraudFlag = '''CASE WHEN FlagType = "Fraud" THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('FraudFlag', 'CycleFlag', '{}','Rule to check Fraud flag for shiptoaccount and cycledate', 6,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(FraudFlag,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)

MapFlag = '''CASE WHEN FlagType = "Map" THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('MapFlag', 'CycleFlag', '{}','Rule to check Map Flag for shiptoaccount and cycledate', 7,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(MapFlag,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)

TNCFlag = '''CASE WHEN FlagType = "TNC" THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('TNCFlag', 'CycleFlag', '{}','Rule to check TNC Flag for shiptoaccount and cycledate', 8,'{}','', 
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(TNCFlag,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)

OffBoarding_NonP3Flag = '''CASE WHEN FlagType = "OffBoarding_NonP3" THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('OffBoarding_NonP3Flag', 'CycleFlag', '{}','Rule to check OffBoarding_NonP3 Flag for shiptoaccount and cycledate', 9,'{}','', 
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(OffBoarding_NonP3Flag ,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)

OffBoarding_NonP3_TreatmentFlag = '''CASE WHEN FlagType = "OffBoarding_NonP3_Treatment" THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('OffBoarding_NonP3_TreatmentFlag', 'CycleFlag', '{}','Rule to check OffBoarding_NonP3_Treatment Flag for shiptoaccount and cycledate', 10,'{}','', 
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(OffBoarding_NonP3_TreatmentFlag ,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)

TNCPerCycleFlag = '''CASE WHEN FlagType = "TNC" AND ConsumerSubscriptionUUID IS NULL THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('TNCPerCycleFlag', 'CycleFlag', '{}','Rule to check TNCPerCycleFlag Flag for shiptoaccount and cycledate', 11,'{}','', 
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(TNCPerCycleFlag,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)

OffBoarding_NonP3PerCycleFlag = '''CASE WHEN FlagType = "OffBoarding_NonP3" AND ConsumerSubscriptionUUID IS NULL THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('OffBoarding_NonP3PerCycleFlag', 'CycleFlag', '{}','Rule to check OffBoarding_NonP3PerCycleFlag Flag for shiptoaccount and cycledate', 12,'{}','', 
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(OffBoarding_NonP3PerCycleFlag,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)

OffBoarding_NonP3_TreatmentPerCycleFlag = '''CASE WHEN FlagType = "OffBoarding_NonP3_Treatment" AND ConsumerSubscriptionUUID IS NULL THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('OffBoarding_NonP3_TreatmentPerCycleFlag', 'CycleFlag', '{}','Rule to check OffBoarding_NonP3_TreatmentPerCycleFlag Flag for shiptoaccount and cycledate', 13,'{}','', 
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(OffBoarding_NonP3_TreatmentPerCycleFlag,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)

PerCycleFeeFlag = '''CASE WHEN PlanType = "SmallTreatmentPlan" THEN 1 ELSE 0 END '''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('PerCycleFeeFlag', 'CycleFlag', '{}','Rule to check Per Cycle fee based on SmallTreatmentPlan plan type and subscription', 14,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(PerCycleFeeFlag,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)

PerCycleFlag = '''CASE WHEN NoPhoneNumberFlag = 1 THEN 1  WHEN PerCycleFeeFlag = 1 THEN 1 WHEN FlagType is not null AND FlagSubType = "PerCycle" THEN 1 ELSE 0 END '''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('PerCycleFlag', 'CycleFlag', '{}','Rule to check Flag Type is PerCycle for shiptoaccount and cycledate', 15,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(PerCycleFlag,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)

PerCycleWhenNoSubscriptionFlag = '''CASE WHEN PerCycleFeeFlag = 1 THEN 1 WHEN TNCFlag = 1 THEN 1 WHEN OffBoarding_NonP3Flag = 1 THEN 1 WHEN OffBoarding_NonP3_TreatmentFlag = 1 THEN 1 ELSE 0 END '''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('PerCycleWhenNoSubscriptionFlag', 'CycleFlag', '{}','Rule to check if classification that need to wait before any checks', 16,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(PerCycleWhenNoSubscriptionFlag,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)

EfficacyFlag = '''CASE WHEN IsValidEfficacy = "Yes" THEN 1 ELSE 0 END '''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('EfficacyFlag', 'CycleFlag', '{}','Rule to check if classification that need to wait before any checks', 17,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(EfficacyFlag,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)

NewSubscriptionStartDate = '''CycleDate'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('NewSubscriptionStartDate', 'ConsumerSubscription_Live', '{}','Populate subscription start date based on CycleDate', 1,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(NewSubscriptionStartDate,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)


NewSubscriptionEndDate = '''CASE WHEN ifnull(InvoiceExceptionFlag,"N") = "N" THEN date_add(CycleDate,150) ELSE SubscriptionEndDate END '''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('NewSubscriptionEndDate', 'ConsumerSubscription_Live', '{}','Populate subscription end date based on CycleDate', 2,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(NewSubscriptionEndDate,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)

PerPatientFeeFlag = '''CASE WHEN PilotSubscriptionFlag = "Y" THEN 0 WHEN ROW_NUM = 1 AND SubscriptionInvoiceAmt = 0 THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('PerPatientFeeFlag', 'InvoiceAddendumDetails_Live', '{}','Calculate PerPatientFeeflag which will be used to calculate classification', 1,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(PerPatientFeeFlag,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)

FollowUpVisitFlag = '''CASE WHEN PilotSubscriptionFlag = "Y" THEN 1 WHEN SubscriptionInvoiceAmt > 0 THEN 1 WHEN SubscriptionInvoiceAmt = 0 AND ROW_NUM > 1 THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('FollowUpVisitFlag', 'InvoiceAddendumDetails_Live', '{}','Calculate FollowUpVisitFlag which will be used to calculate classification', 2,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(FollowUpVisitFlag,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)

PerPatientFeeMapViolationFlag = '''CASE WHEN MapFlag = 1 AND PerPatientFeeFlag = 1 AND NoPhoneNumberFlag = 0 THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('PerPatientFeeMapViolationFlag', 'InvoiceAddendumDetails_Live', '{}','Calculate PerPatientFeeMapViolationFlag which will be used to calculate classification', 3,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(PerPatientFeeMapViolationFlag,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)

TNCViolationFlag = '''CASE WHEN TNCPerCycleFlag = 1 AND NoPhoneNumberFlag = 0 THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('TNCViolationFlag', 'InvoiceAddendumDetails_Live', '{}','Calculate classification based on flags populated earlier', 4,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(TNCViolationFlag,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)

OffBoarding_NonP3ViolationFlag = '''CASE WHEN OffBoarding_NonP3PerCycleFlag = 1 AND NoPhoneNumberFlag = 0 THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('OffBoarding_NonP3ViolationFlag', 'InvoiceAddendumDetails_Live', '{}','Calculate classification based on flags populated earlier', 5,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(OffBoarding_NonP3ViolationFlag ,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)

OffBoarding_NonP3_TreatmentViolationFlag = '''CASE WHEN OffBoarding_NonP3_TreatmentPerCycleFlag = 1 AND NoPhoneNumberFlag = 0 THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('OffBoarding_NonP3_TreatmentViolationFlag', 'InvoiceAddendumDetails_Live', '{}','Calculate classification based on flags populated earlier', 6,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(OffBoarding_NonP3_TreatmentViolationFlag ,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)


PatientClassificationName_PreEfficacyRule = '''CASE WHEN NoPhoneNumberFlag = 1 THEN "NoPatientAssociationFee" WHEN PerCycleFeeFlag = 1 AND ConsumerSubscriptionUUID IS NULL THEN "PerCycleFee" WHEN PerCycleFeeFlag = 1 AND ConsumerSubscriptionUUID IS NOT NULL AND (FraudFlag = 1 OR OffBoarding_NonP3ViolationFlag = 1 OR PerPatientFeeMapViolationFlag = 1)  THEN "PerCycleFee" WHEN FraudFlag = 1 THEN "NonP3PatientFraudViolation" WHEN TNCViolationFlag = 1 THEN "NonP3Fee" WHEN OffBoarding_NonP3ViolationFlag = 1 THEN "NonP3FeeOffboarding" WHEN OffBoarding_NonP3_TreatmentViolationFlag = 1 THEN "NonP3FeeOffboardingTreatment" WHEN PerPatientFeeFlag = 1 THEN "PerPatientFee" WHEN FollowUpVisitFlag = 1 THEN "FollowUpVisit"  ELSE "" END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('PatientClassificationName_PreEfficacyRule', 'InvoiceAddendumDetails_Live', '{}','Calculate classification based on flags populated earlier', 49,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(PatientClassificationName_PreEfficacyRule,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)

PatientClassificationName = '''CASE WHEN PatientClassificationName_PreEfficacyRule NOT IN ("FollowUpVisit","PerPatientFee") AND EfficacyFlag = 0 THEN "PerCycleEfficacyFail" ELSE PatientClassificationName_PreEfficacyRule END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('PatientClassificationName', 'InvoiceAddendumDetails_Live', '{}','Calculate classification based on flags populated earlier', 50,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(PatientClassificationName,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)



LateCycleFlag = '''CASE WHEN CycleDate <= MaxInvoiceDate THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('LateCycleFlag', 'InvoiceAddendumDetails_Live', '{}','Rule to check Late Cycle which needs to adjust consumer subscription', 9,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(LateCycleFlag,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)


Comments = '''CASE WHEN PatientClassificationName = "PerCycleEfficacyFail" THEN "PerCycleEfficacyFail" WHEN LateCycleFlag = 1 AND PerCycleFeeFlag = 1 AND ConsumerSubscriptionUUID IS NULL AND NoPhoneNumberFlag = 0 THEN "LateCycle_PerCycleFee" WHEN LateCycleFlag = 1 AND PerCycleFeeFlag = 1 AND ConsumerSubscriptionUUID IS NOT NULL AND NoPhoneNumberFlag = 0 AND (FraudFlag = 1 OR OffBoarding_NonP3ViolationFlag = 1 OR PerPatientFeeMapViolationFlag = 1) THEN "LateCycle_PerCycleFee" WHEN LateCycleFlag = 1 AND FraudFlag = 1 AND NoPhoneNumberFlag = 0 AND PerCycleFeeFlag = 0 THEN "LateCycle_Flag_NonP3PatientFraudViolation" WHEN FraudFlag = 1 AND NoPhoneNumberFlag = 0  AND PerCycleFeeFlag = 0 THEN "Flag_NonP3PatientFraudViolation" WHEN TNCViolationFlag = 1 AND PerCycleFeeFlag = 0 THEN "Flag_NonP3Fee" WHEN OffBoarding_NonP3ViolationFlag = 1  AND PerCycleFeeFlag = 0 THEN "OffBoardingFlag_NonP3Fee" WHEN OffBoarding_NonP3_TreatmentViolationFlag = 1 AND PerCycleFeeFlag = 0 THEN "OffBoardingFlag_NonP3Fee" WHEN LateCycleFlag = 0 THEN "" WHEN NoPhoneNumberFlag = 1 THEN "LateCycle_NoPatientAssociation" WHEN PerPatientFeeFlag = 1 THEN "LateCycle_NewSubscription" WHEN FollowUpVisitFlag = 1 AND ConsumerSubscription_Comments = "LateCycle_SubscriptionAdjust" THEN "LateCycle_FollowUp_Adjust" WHEN FollowUpVisitFlag = 1 THEN "LateCycle_FollowUp" ELSE "" END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('Comments', 'InvoiceAddendumDetails_Live', '{}','Populate comments based on subscriptionshipTo,  SoldTo,  classification  and Latecycle ', 100,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(Comments,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)


IncrementalInvoiceCharge = '''CASE WHEN InvoiceCalculationType = "PerCycle" THEN CycleCount * ListPrice ELSE ListPrice END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('IncrementalInvoiceCharge', 'InvoiceAddendumDetails_Live_InvoiceAmt', '{}','Calculate incremental invoice amount based on NoPhoneNumberFlag and ListPrice', 1,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(IncrementalInvoiceCharge,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)


Comments = '''CASE WHEN AdjustSubscriptionFlag = 1 THEN "LateCycle_SubscriptionAdjust" ELSE "" END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('Comments', 'AdjustSubscription_TruUp', '{}','Populate comments based on subscription adjust flag for latecycles truup', 1,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(Comments,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)



NewSubscriptionStartDate = '''CycleDate'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('NewSubscriptionStartDate', 'AdjustSubscription_TruUp', '{}','Populate new subscriptionstartdate based on latecycle data', 2,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(NewSubscriptionStartDate,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)


NewSubscriptionEndDate = '''CASE WHEN ifnull(InvoiceExceptionFlag,"N") = "N" THEN date_add(CycleDate,150) ELSE SubscriptionEndDate END '''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('NewSubscriptionEndDate', 'AdjustSubscription_TruUp', '{}','Populate new subscriptionstartdate based on latecycle data', 3,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(NewSubscriptionEndDate,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)


RealignNewSubscriptionFlag = '''CASE WHEN VersionID > 1 AND NKEY_ConsumerSubscriptionUUID IS NULL THEN 1 ELSE 0 END '''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('RealignNewSubscriptionFlag', 'RealignNewSubscription_TruUp', '{}','Populate flag to calculate subscription realignment records', 1,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(RealignNewSubscriptionFlag,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)


NewSubscriptionStartDate = '''CycleDate'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('NewSubscriptionStartDate', 'RealignNewSubscription_TruUp', '{}','Populate subscriptionstartdate for realignment records', 2,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(NewSubscriptionStartDate,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)


NewSubscriptionEndDate = '''CASE WHEN ifnull(InvoiceExceptionFlag,"N") = "N" THEN date_add(CycleDate,150) ELSE SubscriptionEndDate END '''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('NewSubscriptionEndDate', 'RealignNewSubscription_TruUp', '{}','Populate subscriptionenddate for realignment records', 3,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(NewSubscriptionEndDate,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)

Comments = '''"LateCycle_RealignNewSubscription"'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('Comments', 'RealignNewSubscription_TruUp', '{}','popuate comments for realignment records', 4,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(Comments,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)

PerPatientFeeFlag = '''CASE WHEN IncrementalInvoiceCharge > 0 THEN 1 WHEN ROW_NUM = 1 AND Prev_SubscriptionInvoiceAmt = 0 THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('PerPatientFeeFlag', 'InvoiceAddendumDetails_TruUp', '{}','calculate PerpatinetFeeFlag for all realignment records', 1,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(PerPatientFeeFlag,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)

FollowUpVisitFlag = '''CASE WHEN Prev_SubscriptionInvoiceAmt > 0 THEN 1 WHEN SubscriptionInvoiceAmt = 0 AND ROW_NUM > 1 THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('FollowUpVisitFlag', 'InvoiceAddendumDetails_TruUp', '{}','calculate FollowUpVisitFlag for all realignment records', 2,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(FollowUpVisitFlag,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)

MapFlag = '''CASE WHEN FlagType = "Map" THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('MapFlag', 'InvoiceAddendumDetails_TruUp', '{}','Rule to check Map Flag for shiptoaccount and cycledate', 3,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(MapFlag,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)

PerPatientFeeMapViolationFlag = '''CASE WHEN MapFlag = 1 AND PerPatientFeeFlag = 1 THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('PerPatientFeeMapViolationFlag', 'InvoiceAddendumDetails_TruUp', '{}','calculate FollowUpVisitFlag for all realignment records', 4,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(PerPatientFeeMapViolationFlag,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)

PatientClassificationName = '''CASE WHEN PerPatientFeeFlag = 1 THEN "PerPatientFee"  WHEN FollowUpVisitFlag = 1 THEN "FollowUpVisit" ELSE "" END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('PatientClassificationName', 'InvoiceAddendumDetails_TruUp', '{}','populate classification based on flags calculated earlierfor all realignment records', 5,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(PatientClassificationName,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)

LateCycleFlag = '''CASE WHEN CycleDate <= MaxInvoiceDate THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('LateCycleFlag', 'InvoiceAddendumDetails_TruUp', '{}','Rule to check Late Cycle which needs to adjust consumer subscription', 6,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(LateCycleFlag,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)

Comments = '''CASE WHEN LateCycleFlag = 0 THEN "" WHEN Comments_NKEY IS NOT NULL THEN Comments_NKEY WHEN PerPatientFeeFlag = 1 THEN "LateCycle_NewSubscription" WHEN FollowUpVisitFlag = 1 THEN "LateCycle_FollowUp" ELSE "" END '''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('Comments', 'InvoiceAddendumDetails_TruUp', '{}','Populate comments for all realignment records', 7,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(Comments,PromotionUUID_US)
print(s_sql)
spark.sql(s_sql)



In [0]:
display(spark.read.format('delta').table('promotion.DIM_Promotion'))

In [0]:
PromotionName = '4P Beta CA'
PromotionUUID_CA = (spark.read.format('delta').table('promotion.DIM_Promotion'))\
    .select("PromotionUUID")\
    .filter(col("PromotionName") == f"{PromotionName}" )\
    .head()[0]

In [0]:
Comments = '''CASE WHEN IsInvoiced = 0 AND ExceptionCreditAmount = 950 THEN "Exception_PerPatientPreInvoice" WHEN IsInvoiced = 1 AND ExceptionCreditAmount = 950 THEN "Exception_PerPatientPostInvoice" WHEN ExceptionCreditAmount = 0 THEN "Exception_PerPatientPostInvoice" WHEN IsInvoiced = 0 AND ExceptionCreditAmount = CycleCount * 200 THEN "Exception_PerCycleFeePreInvoice" WHEN IsInvoiced = 1 AND ExceptionCreditAmount = CycleCount * 200 THEN "Exception_PerCycleFeePostInvoice" WHEN IsInvoiced = 0 AND ExceptionCreditAmount <> 950 THEN "Exception_NoPatientPreInvoice" WHEN IsInvoiced = 1 AND ExceptionCreditAmount <> 950 THEN "Exception_NoPatientPostInvoice" END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('Comments', 'InvoiceException', '{}','Check exception qualification PerpatientFee-NoPatientFee and   PreInvoice-PostInvoice and populate respective comments', 1,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(Comments,PromotionUUID_CA)
print(s_sql)
spark.sql(s_sql)

NewSubscriptionEndDate = '''CASE WHEN SubscriptionEndDate > InvoiceException_CreatedDate THEN InvoiceException_CreatedDate ELSE SubscriptionEndDate END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('NewSubscriptionEndDate', 'InvoiceException', '{}','Calculate subscriptionEndDate based on exception date', 2,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(NewSubscriptionEndDate,PromotionUUID_CA)
print(s_sql)
spark.sql(s_sql)


PatientClassificationName = '''CASE WHEN IsInvoiced = 0 AND ExceptionCreditAmount = 950 THEN "PerPatientFeeExceptionPreInvoice" WHEN IsInvoiced = 1 AND ExceptionCreditAmount = 950 THEN "PerPatientFeeExceptionPostInvoice" WHEN IsInvoiced = 0 AND ExceptionCreditAmount = CycleCount * 200 THEN "PerCycleFeeExceptionPreInvoice" WHEN IsInvoiced = 1 AND ExceptionCreditAmount = CycleCount * 200 THEN "PerCycleFeeExceptionPostInvoice" WHEN IsInvoiced = 0 AND ExceptionCreditAmount <> 950 THEN "NoPatientAssociationFeeExceptionPreInvoice" WHEN IsInvoiced = 1 AND ExceptionCreditAmount <> 950 THEN "NoPatientAssociationFeeExceptionPostInvoice" END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('PatientClassificationName', 'InvoiceException', '{}','Check exception qualification PerpatientFee-NoPatientFee and PreInvoice-PostInvoice and populate respective classifaction', 3,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(PatientClassificationName,PromotionUUID_CA)
print(s_sql)
spark.sql(s_sql)

CoolSculptingID = '''CASE WHEN coalesce(CAST(CoolSculptingID AS BIGINT),0) = 0 THEN "MISSING" ELSE CoolSculptingID END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('CoolSculptingID', 'CycleFlag', '{}','Transform CoolScultingID to MISSING for all invalid values', 1,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(CoolSculptingID,PromotionUUID_CA)
print(s_sql)
spark.sql(s_sql)

NewSubscriptionFlag = '''CASE WHEN ConsumerSubscriptionUUID IS NULL THEN 1 ELSE 0 END '''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('NewSubscriptionFlag', 'CycleFlag', '{}','Rule to check New Subscription for Live Data', 2,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(NewSubscriptionFlag,PromotionUUID_CA)
print(s_sql)
spark.sql(s_sql)

AdjustSubscriptionFlag = '''CASE WHEN PilotSubscriptionFlag = "Y" THEN 0 WHEN CycleDate > PreviousSubscriptionEndDate AND CycleDate < SubscriptionStartDate THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('AdjustSubscriptionFlag', 'CycleFlag', '{}','Rule to check Late Cycle which needs to adjust consumer subscription', 3,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(AdjustSubscriptionFlag,PromotionUUID_CA)
print(s_sql)
spark.sql(s_sql)

NoPhoneNumberFlag = '''CASE WHEN CoolSculptingID IS NULL OR CoolSculptingID IN ("UNKNOWN","MISSING","","0000000000") THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('NoPhoneNumberFlag', 'CycleFlag', '{}','Rule to check for CoolsculptingID and determins if NoPhoneNumberFlag', 4,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(NoPhoneNumberFlag,PromotionUUID_CA)
print(s_sql)
spark.sql(s_sql)

LateCycleFlag = '''CASE WHEN CycleDate <= MaxInvoiceDate THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('LateCycleFlag', 'CycleFlag', '{}','Rule to check Late Cycle which needs to adjust consumer subscription', 5,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(LateCycleFlag,PromotionUUID_CA)
print(s_sql)
spark.sql(s_sql)


PerCycleFeeFlag = '''CASE WHEN PlanType = "SmallTreatmentPlan" THEN 1 ELSE 0 END '''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('PerCycleFeeFlag', 'CycleFlag', '{}','Rule to check Per Cycle fee based on SmallTreatmentPlan plan type and subscription', 14,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(PerCycleFeeFlag,PromotionUUID_CA)
print(s_sql)
spark.sql(s_sql)

PerCycleFlag = '''CASE WHEN NoPhoneNumberFlag = 1 THEN 1 WHEN PerCycleFeeFlag = 1 THEN 1 WHEN FlagType is not null AND FlagSubType = "PerCycle" THEN 1 ELSE 0 END '''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('PerCycleFlag', 'CycleFlag', '{}','Rule to check Flag Type is PerCycle for shiptoaccount and cycledate', 15,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(PerCycleFlag,PromotionUUID_CA)
print(s_sql)
spark.sql(s_sql)

PerCycleWhenNoSubscriptionFlag = '''CASE WHEN PerCycleFeeFlag = 1 THEN 1 ELSE 0 END '''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('PerCycleWhenNoSubscriptionFlag', 'CycleFlag', '{}','Rule to check if classification that need to wait before any checks', 16,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(PerCycleWhenNoSubscriptionFlag,PromotionUUID_CA)
print(s_sql)
spark.sql(s_sql)

EfficacyFlag = '''CASE WHEN 1=1 THEN 1 ELSE 0 END '''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('EfficacyFlag', 'CycleFlag', '{}','Rule to check if classification that need to wait before any checks', 17,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(EfficacyFlag,PromotionUUID_CA)
print(s_sql)
spark.sql(s_sql)

NewSubscriptionStartDate = '''CycleDate'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('NewSubscriptionStartDate', 'ConsumerSubscription_Live', '{}','Populate subscription start date based on CycleDate', 1,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(NewSubscriptionStartDate,PromotionUUID_CA)
print(s_sql)
spark.sql(s_sql)


NewSubscriptionEndDate = '''CASE WHEN ifnull(InvoiceExceptionFlag,"N") = "N" THEN date_add(CycleDate,150) ELSE SubscriptionEndDate END '''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('NewSubscriptionEndDate', 'ConsumerSubscription_Live', '{}','Populate subscription end date based on CycleDate', 2,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(NewSubscriptionEndDate,PromotionUUID_CA)
print(s_sql)
spark.sql(s_sql)

PerPatientFeeFlag = '''CASE WHEN PilotSubscriptionFlag = "Y" THEN 0 WHEN ROW_NUM = 1 AND SubscriptionInvoiceAmt = 0 THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('PerPatientFeeFlag', 'InvoiceAddendumDetails_Live', '{}','Calculate PerPatientFeeflag which will be used to calculate classification', 1,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(PerPatientFeeFlag,PromotionUUID_CA)
print(s_sql)
spark.sql(s_sql)

FollowUpVisitFlag = '''CASE WHEN PilotSubscriptionFlag = "Y" THEN 1 WHEN SubscriptionInvoiceAmt > 0 THEN 1 WHEN SubscriptionInvoiceAmt = 0 AND ROW_NUM > 1 THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('FollowUpVisitFlag', 'InvoiceAddendumDetails_Live', '{}','Calculate FollowUpVisitFlag which will be used to calculate classification', 2,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(FollowUpVisitFlag,PromotionUUID_CA)
print(s_sql)
spark.sql(s_sql)



PatientClassificationName = '''CASE WHEN NoPhoneNumberFlag = 1 THEN "NoPatientAssociationFee" WHEN PerCycleFeeFlag = 1 AND FollowUpVisitFlag = 0 AND NoPhoneNumberFlag = 0 THEN "PerCycleFee"  WHEN PerPatientFeeFlag = 1 THEN "PerPatientFee" WHEN FollowUpVisitFlag = 1 THEN "FollowUpVisit"  ELSE "" END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('PatientClassificationName', 'InvoiceAddendumDetails_Live', '{}','Calculate classification based on flags populated earlier', 3,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(PatientClassificationName,PromotionUUID_CA)
print(s_sql)
spark.sql(s_sql)

LateCycleFlag = '''CASE WHEN CycleDate <= MaxInvoiceDate THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('LateCycleFlag', 'InvoiceAddendumDetails_Live', '{}','Rule to check Late Cycle which needs to adjust consumer subscription', 4,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(LateCycleFlag,PromotionUUID_CA)
print(s_sql)
spark.sql(s_sql)

Comments = '''CASE WHEN LateCycleFlag = 1 AND PerCycleFeeFlag = 1 AND FollowUpVisitFlag = 0 AND NoPhoneNumberFlag = 0 THEN "LateCycle_PerCycleFee" WHEN SubscriptionShipToAccountUUID <> ShipToAccountUUID  AND FollowUpVisitFlag = 1 AND LateCycleFlag = 1 THEN "DifferentShipToID_LateCycle_FollowUp" WHEN SubscriptionShipToAccountUUID <> ShipToAccountUUID AND FollowUpVisitFlag = 1 AND LateCycleFlag = 0 THEN "DifferentShipToID_FollowUp" WHEN LateCycleFlag = 0 THEN "" WHEN NoPhoneNumberFlag = 1 THEN "LateCycle_NoPatientAssociation" WHEN PerPatientFeeFlag = 1 THEN "LateCycle_NewSubscription" WHEN FollowUpVisitFlag = 1 AND CycleDate = SubscriptionStartDate AND ConsumerSubscription_Comments = "LateCycle_SubscriptionAdjust" THEN "LateCycle_FollowUp_Adjust" WHEN FollowUpVisitFlag = 1 THEN "LateCycle_FollowUp" ELSE "" END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('Comments', 'InvoiceAddendumDetails_Live', '{}','Populate comments based on subscriptionshipTo,  SoldTo,  classification  and Latecycle ', 5,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(Comments,PromotionUUID_CA)
print(s_sql)
spark.sql(s_sql)


IncrementalInvoiceCharge = '''CASE WHEN InvoiceCalculationType = "PerCycle" THEN CycleCount * ListPrice ELSE ListPrice END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('IncrementalInvoiceCharge', 'InvoiceAddendumDetails_Live_InvoiceAmt', '{}','Calculate incremental invoice amount based on NoPhoneNumberFlag and ListPrice', 1,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(IncrementalInvoiceCharge,PromotionUUID_CA)
print(s_sql)
spark.sql(s_sql)


Comments = '''CASE WHEN AdjustSubscriptionFlag = 1 THEN "LateCycle_SubscriptionAdjust" ELSE "" END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('Comments', 'AdjustSubscription_TruUp', '{}','Populate comments based on subscription adjust flag for latecycles truup', 1,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(Comments,PromotionUUID_CA)
print(s_sql)
spark.sql(s_sql)



NewSubscriptionStartDate = '''CycleDate'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('NewSubscriptionStartDate', 'AdjustSubscription_TruUp', '{}','Populate new subscriptionstartdate based on latecycle data', 2,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(NewSubscriptionStartDate,PromotionUUID_CA)
print(s_sql)
spark.sql(s_sql)


NewSubscriptionEndDate = '''CASE WHEN ifnull(InvoiceExceptionFlag,"N") = "N" THEN date_add(CycleDate,150) ELSE SubscriptionEndDate END '''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('NewSubscriptionEndDate', 'AdjustSubscription_TruUp', '{}','Populate new subscriptionstartdate based on latecycle data', 3,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(NewSubscriptionEndDate,PromotionUUID_CA)
print(s_sql)
spark.sql(s_sql)


RealignNewSubscriptionFlag = '''CASE WHEN VersionID > 1 AND NKEY_ConsumerSubscriptionUUID IS NULL THEN 1 ELSE 0 END '''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('RealignNewSubscriptionFlag', 'RealignNewSubscription_TruUp', '{}','Populate flag to calculate subscription realignment records', 1,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(RealignNewSubscriptionFlag,PromotionUUID_CA)
print(s_sql)
spark.sql(s_sql)


NewSubscriptionStartDate = '''CycleDate'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('NewSubscriptionStartDate', 'RealignNewSubscription_TruUp', '{}','Populate subscriptionstartdate for realignment records', 2,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(NewSubscriptionStartDate,PromotionUUID_CA)
print(s_sql)
spark.sql(s_sql)


NewSubscriptionEndDate = '''CASE WHEN ifnull(InvoiceExceptionFlag,"N") = "N" THEN date_add(CycleDate,150) ELSE SubscriptionEndDate END '''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('NewSubscriptionEndDate', 'RealignNewSubscription_TruUp', '{}','Populate subscriptionenddate for realignment records', 3,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(NewSubscriptionEndDate,PromotionUUID_CA)
print(s_sql)
spark.sql(s_sql)

Comments = '''"LateCycle_RealignNewSubscription"'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('Comments', 'RealignNewSubscription_TruUp', '{}','popuate comments for realignment records', 4,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(Comments,PromotionUUID_CA)
print(s_sql)
spark.sql(s_sql)


PerPatientFeeFlag = '''CASE WHEN IncrementalInvoiceCharge > 0 THEN 1 WHEN ROW_NUM = 1 AND Prev_SubscriptionInvoiceAmt = 0 THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('PerPatientFeeFlag', 'InvoiceAddendumDetails_TruUp', '{}','calculate PerpatinetFeeFlag for all realignment records', 1,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(PerPatientFeeFlag,PromotionUUID_CA)
print(s_sql)
spark.sql(s_sql)

FollowUpVisitFlag = '''CASE WHEN Prev_SubscriptionInvoiceAmt > 0 THEN 1 WHEN SubscriptionInvoiceAmt = 0 AND ROW_NUM > 1 THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('FollowUpVisitFlag', 'InvoiceAddendumDetails_TruUp', '{}','calculate FollowUpVisitFlag for all realignment records', 2,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(FollowUpVisitFlag,PromotionUUID_CA)
print(s_sql)
spark.sql(s_sql)

PatientClassificationName = '''CASE WHEN PerPatientFeeFlag = 1 THEN "PerPatientFee" WHEN FollowUpVisitFlag = 1 THEN "FollowUpVisit" ELSE "" END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('PatientClassificationName', 'InvoiceAddendumDetails_TruUp', '{}','populate classification based on flags calculated earlierfor all realignment records', 3,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(PatientClassificationName,PromotionUUID_CA)
print(s_sql)
spark.sql(s_sql)

LateCycleFlag = '''CASE WHEN CycleDate <= MaxInvoiceDate THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('LateCycleFlag', 'InvoiceAddendumDetails_TruUp', '{}','Rule to check Late Cycle which needs to adjust consumer subscription', 4,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(LateCycleFlag,PromotionUUID_CA)
print(s_sql)
spark.sql(s_sql)

Comments = '''CASE WHEN LateCycleFlag = 0 THEN "" WHEN Comments_NKEY IS NOT NULL THEN Comments_NKEY WHEN PerPatientFeeFlag = 1 THEN "LateCycle_NewSubscription" WHEN FollowUpVisitFlag = 1 THEN "LateCycle_FollowUp" ELSE "" END '''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('Comments', 'InvoiceAddendumDetails_TruUp', '{}','Populate comments for all realignment records', 5,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(Comments,PromotionUUID_CA)
print(s_sql)
spark.sql(s_sql)



In [0]:
PromotionName = '4P Beta UK'
PromotionUUID_UK = (spark.read.format('delta').table('promotion.DIM_Promotion'))\
    .select("PromotionUUID")\
    .filter(col("PromotionName") == f"{PromotionName}" )\
    .head()[0]

In [0]:
Comments = '''CASE WHEN IsInvoiced = 0 AND ExceptionCreditAmount = 950 THEN "Exception_PerPatientPreInvoice" WHEN IsInvoiced = 1 AND ExceptionCreditAmount = 950 THEN "Exception_PerPatientPostInvoice" WHEN ExceptionCreditAmount = 0 THEN "Exception_PerPatientPostInvoice" WHEN IsInvoiced = 0 AND ExceptionCreditAmount = CycleCount * 200 THEN "Exception_PerCycleFeePreInvoice" WHEN IsInvoiced = 1 AND ExceptionCreditAmount = CycleCount * 200 THEN "Exception_PerCycleFeePostInvoice" WHEN IsInvoiced = 0 AND ExceptionCreditAmount <> 950 THEN "Exception_NoPatientPreInvoice" WHEN IsInvoiced = 1 AND ExceptionCreditAmount <> 950 THEN "Exception_NoPatientPostInvoice" END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('Comments', 'InvoiceException', '{}','Check exception qualification PerpatientFee-NoPatientFee and   PreInvoice-PostInvoice and populate respective comments', 1,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(Comments,PromotionUUID_UK)
print(s_sql)
spark.sql(s_sql)

NewSubscriptionEndDate = '''CASE WHEN SubscriptionEndDate > InvoiceException_CreatedDate THEN InvoiceException_CreatedDate ELSE SubscriptionEndDate END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('NewSubscriptionEndDate', 'InvoiceException', '{}','Calculate subscriptionEndDate based on exception date', 2,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(NewSubscriptionEndDate,PromotionUUID_UK)
print(s_sql)
spark.sql(s_sql)


PatientClassificationName = '''CASE WHEN IsInvoiced = 0 AND ExceptionCreditAmount = 950 THEN "PerPatientFeeExceptionPreInvoice" WHEN IsInvoiced = 1 AND ExceptionCreditAmount = 950 THEN "PerPatientFeeExceptionPostInvoice" WHEN IsInvoiced = 0 AND ExceptionCreditAmount = CycleCount * 200 THEN "PerCycleFeeExceptionPreInvoice" WHEN IsInvoiced = 1 AND ExceptionCreditAmount = CycleCount * 200 THEN "PerCycleFeeExceptionPostInvoice" WHEN IsInvoiced = 0 AND ExceptionCreditAmount <> 950 THEN "NoPatientAssociationFeeExceptionPreInvoice" WHEN IsInvoiced = 1 AND ExceptionCreditAmount <> 950 THEN "NoPatientAssociationFeeExceptionPostInvoice" END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('PatientClassificationName', 'InvoiceException', '{}','Check exception qualification PerpatientFee-NoPatientFee and PreInvoice-PostInvoice and populate respective classifaction', 3,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(PatientClassificationName,PromotionUUID_UK)
print(s_sql)
spark.sql(s_sql)

CoolSculptingID = '''CASE WHEN coalesce(CAST(CoolSculptingID AS BIGINT),0) = 0 THEN "MISSING" ELSE CoolSculptingID END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('CoolSculptingID', 'CycleFlag', '{}','Transform CoolScultingID to MISSING for all invalid values', 1,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(CoolSculptingID,PromotionUUID_UK)
print(s_sql)
spark.sql(s_sql)

NewSubscriptionFlag = '''CASE WHEN ConsumerSubscriptionUUID IS NULL THEN 1 ELSE 0 END '''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('NewSubscriptionFlag', 'CycleFlag', '{}','Rule to check New Subscription for Live Data', 2,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(NewSubscriptionFlag,PromotionUUID_UK)
print(s_sql)
spark.sql(s_sql)

AdjustSubscriptionFlag = '''CASE WHEN PilotSubscriptionFlag = "Y" THEN 0 WHEN CycleDate > PreviousSubscriptionEndDate AND CycleDate < SubscriptionStartDate THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('AdjustSubscriptionFlag', 'CycleFlag', '{}','Rule to check Late Cycle which needs to adjust consumer subscription', 3,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(AdjustSubscriptionFlag,PromotionUUID_UK)
print(s_sql)
spark.sql(s_sql)

NoPhoneNumberFlag = '''CASE WHEN CoolSculptingID IS NULL OR CoolSculptingID IN ("UNKNOWN","MISSING","","0000000000") THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('NoPhoneNumberFlag', 'CycleFlag', '{}','Rule to check for CoolsculptingID and determins if NoPhoneNumberFlag', 4,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(NoPhoneNumberFlag,PromotionUUID_UK)
print(s_sql)
spark.sql(s_sql)

LateCycleFlag = '''CASE WHEN CycleDate <= MaxInvoiceDate THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('LateCycleFlag', 'CycleFlag', '{}','Rule to check Late Cycle which needs to adjust consumer subscription', 5,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(LateCycleFlag,PromotionUUID_UK)
print(s_sql)
spark.sql(s_sql)


PerCycleFeeFlag = '''CASE WHEN PlanType = "SmallTreatmentPlan" THEN 1 ELSE 0 END '''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('PerCycleFeeFlag', 'CycleFlag', '{}','Rule to check Per Cycle fee based on SmallTreatmentPlan plan type and subscription', 14,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(PerCycleFeeFlag,PromotionUUID_UK)
print(s_sql)
spark.sql(s_sql)

PerCycleFlag = '''CASE WHEN NoPhoneNumberFlag = 1 THEN 1 WHEN PerCycleFeeFlag = 1 THEN 1 WHEN FlagType is not null AND FlagSubType = "PerCycle" THEN 1 ELSE 0 END '''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('PerCycleFlag', 'CycleFlag', '{}','Rule to check Flag Type is PerCycle for shiptoaccount and cycledate', 15,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(PerCycleFlag,PromotionUUID_UK)
print(s_sql)
spark.sql(s_sql)

PerCycleWhenNoSubscriptionFlag = '''CASE WHEN PerCycleFeeFlag = 1 THEN 1 ELSE 0 END '''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('PerCycleWhenNoSubscriptionFlag', 'CycleFlag', '{}','Rule to check if classification that need to wait before any checks', 16,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(PerCycleWhenNoSubscriptionFlag,PromotionUUID_UK)
print(s_sql)
spark.sql(s_sql)

EfficacyFlag = '''CASE WHEN 1=1 THEN 1 ELSE 0 END '''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('EfficacyFlag', 'CycleFlag', '{}','Rule to check if classification that need to wait before any checks', 17,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(EfficacyFlag,PromotionUUID_UK)
print(s_sql)
spark.sql(s_sql)

NewSubscriptionStartDate = '''CycleDate'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('NewSubscriptionStartDate', 'ConsumerSubscription_Live', '{}','Populate subscription start date based on CycleDate', 1,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(NewSubscriptionStartDate,PromotionUUID_UK)
print(s_sql)
spark.sql(s_sql)


NewSubscriptionEndDate = '''CASE WHEN ifnull(InvoiceExceptionFlag,"N") = "N" THEN date_add(CycleDate,150) ELSE SubscriptionEndDate END '''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('NewSubscriptionEndDate', 'ConsumerSubscription_Live', '{}','Populate subscription end date based on CycleDate', 2,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(NewSubscriptionEndDate,PromotionUUID_UK)
print(s_sql)
spark.sql(s_sql)

PerPatientFeeFlag = '''CASE WHEN PilotSubscriptionFlag = "Y" THEN 0 WHEN ROW_NUM = 1 AND SubscriptionInvoiceAmt = 0 THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('PerPatientFeeFlag', 'InvoiceAddendumDetails_Live', '{}','Calculate PerPatientFeeflag which will be used to calculate classification', 1,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(PerPatientFeeFlag,PromotionUUID_UK)
print(s_sql)
spark.sql(s_sql)

FollowUpVisitFlag = '''CASE WHEN PilotSubscriptionFlag = "Y" THEN 1 WHEN SubscriptionInvoiceAmt > 0 THEN 1 WHEN SubscriptionInvoiceAmt = 0 AND ROW_NUM > 1 THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('FollowUpVisitFlag', 'InvoiceAddendumDetails_Live', '{}','Calculate FollowUpVisitFlag which will be used to calculate classification', 2,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(FollowUpVisitFlag,PromotionUUID_UK)
print(s_sql)
spark.sql(s_sql)



PatientClassificationName = '''CASE WHEN NoPhoneNumberFlag = 1 THEN "NoPatientAssociationFee" WHEN PerCycleFeeFlag = 1 AND FollowUpVisitFlag = 0 AND NoPhoneNumberFlag = 0 THEN "PerCycleFee" WHEN PerPatientFeeFlag = 1 THEN "PerPatientFee" WHEN FollowUpVisitFlag = 1 THEN "FollowUpVisit"  ELSE "" END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('PatientClassificationName', 'InvoiceAddendumDetails_Live', '{}','Calculate classification based on flags populated earlier', 3,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(PatientClassificationName,PromotionUUID_UK)
print(s_sql)
spark.sql(s_sql)

LateCycleFlag = '''CASE WHEN CycleDate <= MaxInvoiceDate THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('LateCycleFlag', 'InvoiceAddendumDetails_Live', '{}','Rule to check Late Cycle which needs to adjust consumer subscription', 4,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(LateCycleFlag,PromotionUUID_UK)
print(s_sql)
spark.sql(s_sql)

Comments = '''CASE WHEN LateCycleFlag = 1 AND PerCycleFeeFlag = 1 AND FollowUpVisitFlag = 0 AND NoPhoneNumberFlag = 0 THEN "LateCycle_PerCycleFee" WHEN SubscriptionShipToAccountUUID <> ShipToAccountUUID  AND FollowUpVisitFlag = 1 AND LateCycleFlag = 1 THEN "DifferentShipToID_LateCycle_FollowUp" WHEN SubscriptionShipToAccountUUID <> ShipToAccountUUID AND FollowUpVisitFlag = 1 AND LateCycleFlag = 0 THEN "DifferentShipToID_FollowUp" WHEN LateCycleFlag = 0 THEN "" WHEN NoPhoneNumberFlag = 1 THEN "LateCycle_NoPatientAssociation" WHEN PerPatientFeeFlag = 1 THEN "LateCycle_NewSubscription" WHEN FollowUpVisitFlag = 1 AND CycleDate = SubscriptionStartDate AND ConsumerSubscription_Comments = "LateCycle_SubscriptionAdjust" THEN "LateCycle_FollowUp_Adjust" WHEN FollowUpVisitFlag = 1 THEN "LateCycle_FollowUp" ELSE "" END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('Comments', 'InvoiceAddendumDetails_Live', '{}','Populate comments based on subscriptionshipTo,  SoldTo,  classification  and Latecycle ', 5,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(Comments,PromotionUUID_UK)
print(s_sql)
spark.sql(s_sql)

IncrementalInvoiceCharge = '''CASE WHEN InvoiceCalculationType = "PerCycle" THEN CycleCount * ListPrice ELSE ListPrice END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('IncrementalInvoiceCharge', 'InvoiceAddendumDetails_Live_InvoiceAmt', '{}','Calculate incremental invoice amount based on NoPhoneNumberFlag and ListPrice', 1,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(IncrementalInvoiceCharge,PromotionUUID_UK)
print(s_sql)
spark.sql(s_sql)


Comments = '''CASE WHEN AdjustSubscriptionFlag = 1 THEN "LateCycle_SubscriptionAdjust" ELSE "" END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('Comments', 'AdjustSubscription_TruUp', '{}','Populate comments based on subscription adjust flag for latecycles truup', 1,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(Comments,PromotionUUID_UK)
print(s_sql)
spark.sql(s_sql)



NewSubscriptionStartDate = '''CycleDate'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('NewSubscriptionStartDate', 'AdjustSubscription_TruUp', '{}','Populate new subscriptionstartdate based on latecycle data', 2,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(NewSubscriptionStartDate,PromotionUUID_UK)
print(s_sql)
spark.sql(s_sql)


NewSubscriptionEndDate = '''CASE WHEN ifnull(InvoiceExceptionFlag,"N") = "N" THEN date_add(CycleDate,150) ELSE SubscriptionEndDate END '''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('NewSubscriptionEndDate', 'AdjustSubscription_TruUp', '{}','Populate new subscriptionstartdate based on latecycle data', 3,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(NewSubscriptionEndDate,PromotionUUID_UK)
print(s_sql)
spark.sql(s_sql)


RealignNewSubscriptionFlag = '''CASE WHEN VersionID > 1 AND NKEY_ConsumerSubscriptionUUID IS NULL THEN 1 ELSE 0 END '''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('RealignNewSubscriptionFlag', 'RealignNewSubscription_TruUp', '{}','Populate flag to calculate subscription realignment records', 1,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(RealignNewSubscriptionFlag,PromotionUUID_UK)
print(s_sql)
spark.sql(s_sql)


NewSubscriptionStartDate = '''CycleDate'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('NewSubscriptionStartDate', 'RealignNewSubscription_TruUp', '{}','Populate subscriptionstartdate for realignment records', 2,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(NewSubscriptionStartDate,PromotionUUID_UK)
print(s_sql)
spark.sql(s_sql)


NewSubscriptionEndDate = '''CASE WHEN ifnull(InvoiceExceptionFlag,"N") = "N" THEN date_add(CycleDate,150) ELSE SubscriptionEndDate END '''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('NewSubscriptionEndDate', 'RealignNewSubscription_TruUp', '{}','Populate subscriptionenddate for realignment records', 3,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(NewSubscriptionEndDate,PromotionUUID_UK)
print(s_sql)
spark.sql(s_sql)

Comments = '''"LateCycle_RealignNewSubscription"'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('Comments', 'RealignNewSubscription_TruUp', '{}','popuate comments for realignment records', 4,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(Comments,PromotionUUID_UK)
print(s_sql)
spark.sql(s_sql)


PerPatientFeeFlag = '''CASE WHEN IncrementalInvoiceCharge > 0 THEN 1 WHEN ROW_NUM = 1 AND Prev_SubscriptionInvoiceAmt = 0 THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('PerPatientFeeFlag', 'InvoiceAddendumDetails_TruUp', '{}','calculate PerpatinetFeeFlag for all realignment records', 1,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(PerPatientFeeFlag,PromotionUUID_UK)
print(s_sql)
spark.sql(s_sql)

FollowUpVisitFlag = '''CASE WHEN Prev_SubscriptionInvoiceAmt > 0 THEN 1 WHEN SubscriptionInvoiceAmt = 0 AND ROW_NUM > 1 THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('FollowUpVisitFlag', 'InvoiceAddendumDetails_TruUp', '{}','calculate FollowUpVisitFlag for all realignment records', 2,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(FollowUpVisitFlag,PromotionUUID_UK)
print(s_sql)
spark.sql(s_sql)

PatientClassificationName = '''CASE WHEN PerPatientFeeFlag = 1 THEN "PerPatientFee" WHEN FollowUpVisitFlag = 1 THEN "FollowUpVisit" ELSE "" END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('PatientClassificationName', 'InvoiceAddendumDetails_TruUp', '{}','populate classification based on flags calculated earlierfor all realignment records', 3,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(PatientClassificationName,PromotionUUID_UK)
print(s_sql)
spark.sql(s_sql)

LateCycleFlag = '''CASE WHEN CycleDate <= MaxInvoiceDate THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('LateCycleFlag', 'InvoiceAddendumDetails_TruUp', '{}','Rule to check Late Cycle which needs to adjust consumer subscription', 4,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(LateCycleFlag,PromotionUUID_UK)
print(s_sql)
spark.sql(s_sql)

Comments = '''CASE WHEN LateCycleFlag = 0 THEN "" WHEN Comments_NKEY IS NOT NULL THEN Comments_NKEY WHEN PerPatientFeeFlag = 1 THEN "LateCycle_NewSubscription" WHEN FollowUpVisitFlag = 1 THEN "LateCycle_FollowUp" ELSE "" END '''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('Comments', 'InvoiceAddendumDetails_TruUp', '{}','Populate comments for all realignment records', 5,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(Comments,PromotionUUID_UK)
print(s_sql)
spark.sql(s_sql)



In [0]:
PromotionName = '4P Beta SP'
PromotionUUID_SP = (spark.read.format('delta').table('promotion.DIM_Promotion'))\
    .select("PromotionUUID")\
    .filter(col("PromotionName") == f"{PromotionName}" )\
    .head()[0]

In [0]:
Comments = '''CASE WHEN IsInvoiced = 0 AND ExceptionCreditAmount = 950 THEN "Exception_PerPatientPreInvoice" WHEN IsInvoiced = 1 AND ExceptionCreditAmount = 950 THEN "Exception_PerPatientPostInvoice" WHEN ExceptionCreditAmount = 0 THEN "Exception_PerPatientPostInvoice" WHEN IsInvoiced = 0 AND ExceptionCreditAmount = CycleCount * 200 THEN "Exception_PerCycleFeePreInvoice" WHEN IsInvoiced = 1 AND ExceptionCreditAmount = CycleCount * 200 THEN "Exception_PerCycleFeePostInvoice" WHEN IsInvoiced = 0 AND ExceptionCreditAmount <> 950 THEN "Exception_NoPatientPreInvoice" WHEN IsInvoiced = 1 AND ExceptionCreditAmount <> 950 THEN "Exception_NoPatientPostInvoice" END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('Comments', 'InvoiceException', '{}','Check exception qualification PerpatientFee-NoPatientFee and   PreInvoice-PostInvoice and populate respective comments', 1,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(Comments,PromotionUUID_SP)
print(s_sql)
spark.sql(s_sql)

NewSubscriptionEndDate = '''CASE WHEN SubscriptionEndDate > InvoiceException_CreatedDate THEN InvoiceException_CreatedDate ELSE SubscriptionEndDate END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('NewSubscriptionEndDate', 'InvoiceException', '{}','Calculate subscriptionEndDate based on exception date', 2,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(NewSubscriptionEndDate,PromotionUUID_SP)
print(s_sql)
spark.sql(s_sql)


PatientClassificationName = '''CASE WHEN IsInvoiced = 0 AND ExceptionCreditAmount = 950 THEN "PerPatientFeeExceptionPreInvoice" WHEN IsInvoiced = 1 AND ExceptionCreditAmount = 950 THEN "PerPatientFeeExceptionPostInvoice" WHEN IsInvoiced = 0 AND ExceptionCreditAmount = CycleCount * 200 THEN "PerCycleFeeExceptionPreInvoice" WHEN IsInvoiced = 1 AND ExceptionCreditAmount = CycleCount * 200 THEN "PerCycleFeeExceptionPostInvoice" WHEN IsInvoiced = 0 AND ExceptionCreditAmount <> 950 THEN "NoPatientAssociationFeeExceptionPreInvoice" WHEN IsInvoiced = 1 AND ExceptionCreditAmount <> 950 THEN "NoPatientAssociationFeeExceptionPostInvoice" END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('PatientClassificationName', 'InvoiceException', '{}','Check exception qualification PerpatientFee-NoPatientFee and PreInvoice-PostInvoice and populate respective classifaction', 3,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(PatientClassificationName,PromotionUUID_SP)
print(s_sql)
spark.sql(s_sql)

CoolSculptingID = '''CASE WHEN coalesce(CAST(CoolSculptingID AS BIGINT),0) = 0 THEN "MISSING" ELSE CoolSculptingID END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('CoolSculptingID', 'CycleFlag', '{}','Transform CoolScultingID to MISSING for all invalid values', 1,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(CoolSculptingID,PromotionUUID_SP)
print(s_sql)
spark.sql(s_sql)

NewSubscriptionFlag = '''CASE WHEN ConsumerSubscriptionUUID IS NULL THEN 1 ELSE 0 END '''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('NewSubscriptionFlag', 'CycleFlag', '{}','Rule to check New Subscription for Live Data', 2,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(NewSubscriptionFlag,PromotionUUID_SP)
print(s_sql)
spark.sql(s_sql)

AdjustSubscriptionFlag = '''CASE WHEN PilotSubscriptionFlag = "Y" THEN 0 WHEN CycleDate > PreviousSubscriptionEndDate AND CycleDate < SubscriptionStartDate THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('AdjustSubscriptionFlag', 'CycleFlag', '{}','Rule to check Late Cycle which needs to adjust consumer subscription', 3,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(AdjustSubscriptionFlag,PromotionUUID_SP)
print(s_sql)
spark.sql(s_sql)

NoPhoneNumberFlag = '''CASE WHEN CoolSculptingID IS NULL OR CoolSculptingID IN ("UNKNOWN","MISSING","","0000000000") THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('NoPhoneNumberFlag', 'CycleFlag', '{}','Rule to check for CoolsculptingID and determins if NoPhoneNumberFlag', 4,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(NoPhoneNumberFlag,PromotionUUID_SP)
print(s_sql)
spark.sql(s_sql)

LateCycleFlag = '''CASE WHEN CycleDate <= MaxInvoiceDate THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('LateCycleFlag', 'CycleFlag', '{}','Rule to check Late Cycle which needs to adjust consumer subscription', 5,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(LateCycleFlag,PromotionUUID_SP)
print(s_sql)
spark.sql(s_sql)


PerCycleFeeFlag = '''CASE WHEN PlanType = "SmallTreatmentPlan" THEN 1 ELSE 0 END '''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('PerCycleFeeFlag', 'CycleFlag', '{}','Rule to check Per Cycle fee based on SmallTreatmentPlan plan type and subscription', 14,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(PerCycleFeeFlag,PromotionUUID_SP)
print(s_sql)
spark.sql(s_sql)

PerCycleFlag = '''CASE WHEN NoPhoneNumberFlag = 1 THEN 1 WHEN PerCycleFeeFlag = 1 THEN 1 WHEN FlagType is not null AND FlagSubType = "PerCycle" THEN 1 ELSE 0 END '''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('PerCycleFlag', 'CycleFlag', '{}','Rule to check Flag Type is PerCycle for shiptoaccount and cycledate', 15,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(PerCycleFlag,PromotionUUID_SP)
print(s_sql)
spark.sql(s_sql)

PerCycleWhenNoSubscriptionFlag = '''CASE WHEN PerCycleFeeFlag = 1 THEN 1 ELSE 0 END '''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('PerCycleWhenNoSubscriptionFlag', 'CycleFlag', '{}','Rule to check if classification that need to wait before any checks', 16,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(PerCycleWhenNoSubscriptionFlag,PromotionUUID_SP)
print(s_sql)
spark.sql(s_sql)


EfficacyFlag = '''CASE WHEN 1=1 THEN 1 ELSE 0 END '''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('EfficacyFlag', 'CycleFlag', '{}','Rule to check if classification that need to wait before any checks', 17,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(EfficacyFlag,PromotionUUID_SP)
print(s_sql)
spark.sql(s_sql)

NewSubscriptionStartDate = '''CycleDate'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('NewSubscriptionStartDate', 'ConsumerSubscription_Live', '{}','Populate subscription start date based on CycleDate', 1,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(NewSubscriptionStartDate,PromotionUUID_SP)
print(s_sql)
spark.sql(s_sql)


NewSubscriptionEndDate = '''CASE WHEN ifnull(InvoiceExceptionFlag,"N") = "N" THEN date_add(CycleDate,150) ELSE SubscriptionEndDate END '''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('NewSubscriptionEndDate', 'ConsumerSubscription_Live', '{}','Populate subscription end date based on CycleDate', 2,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(NewSubscriptionEndDate,PromotionUUID_SP)
print(s_sql)
spark.sql(s_sql)

PerPatientFeeFlag = '''CASE WHEN PilotSubscriptionFlag = "Y" THEN 0 WHEN ROW_NUM = 1 AND SubscriptionInvoiceAmt = 0 THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('PerPatientFeeFlag', 'InvoiceAddendumDetails_Live', '{}','Calculate PerPatientFeeflag which will be used to calculate classification', 1,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(PerPatientFeeFlag,PromotionUUID_SP)
print(s_sql)
spark.sql(s_sql)

FollowUpVisitFlag = '''CASE WHEN PilotSubscriptionFlag = "Y" THEN 1 WHEN SubscriptionInvoiceAmt > 0 THEN 1 WHEN SubscriptionInvoiceAmt = 0 AND ROW_NUM > 1 THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('FollowUpVisitFlag', 'InvoiceAddendumDetails_Live', '{}','Calculate FollowUpVisitFlag which will be used to calculate classification', 2,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(FollowUpVisitFlag,PromotionUUID_SP)
print(s_sql)
spark.sql(s_sql)




PatientClassificationName = '''CASE WHEN NoPhoneNumberFlag = 1 THEN "NoPatientAssociationFee" WHEN PerCycleFeeFlag = 1 AND FollowUpVisitFlag = 0 AND NoPhoneNumberFlag = 0 THEN "PerCycleFee" WHEN PerPatientFeeFlag = 1 THEN "PerPatientFee" WHEN FollowUpVisitFlag = 1 THEN "FollowUpVisit"  ELSE "" END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('PatientClassificationName', 'InvoiceAddendumDetails_Live', '{}','Calculate classification based on flags populated earlier', 3,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(PatientClassificationName,PromotionUUID_SP)
print(s_sql)
spark.sql(s_sql)

LateCycleFlag = '''CASE WHEN CycleDate <= MaxInvoiceDate THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('LateCycleFlag', 'InvoiceAddendumDetails_Live', '{}','Rule to check Late Cycle which needs to adjust consumer subscription', 4,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(LateCycleFlag,PromotionUUID_SP)
print(s_sql)
spark.sql(s_sql)

Comments = '''CASE WHEN LateCycleFlag = 1 AND PerCycleFeeFlag = 1 AND FollowUpVisitFlag = 0 AND NoPhoneNumberFlag = 0 THEN "LateCycle_PerCycleFee" WHEN SubscriptionShipToAccountUUID <> ShipToAccountUUID  AND FollowUpVisitFlag = 1 AND LateCycleFlag = 1 THEN "DifferentShipToID_LateCycle_FollowUp" WHEN SubscriptionShipToAccountUUID <> ShipToAccountUUID AND FollowUpVisitFlag = 1 AND LateCycleFlag = 0 THEN "DifferentShipToID_FollowUp" WHEN LateCycleFlag = 0 THEN "" WHEN NoPhoneNumberFlag = 1 THEN "LateCycle_NoPatientAssociation" WHEN PerPatientFeeFlag = 1 THEN "LateCycle_NewSubscription" WHEN FollowUpVisitFlag = 1 AND CycleDate = SubscriptionStartDate AND ConsumerSubscription_Comments = "LateCycle_SubscriptionAdjust"  THEN "LateCycle_FollowUp_Adjust" WHEN FollowUpVisitFlag = 1 THEN "LateCycle_FollowUp" ELSE "" END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('Comments', 'InvoiceAddendumDetails_Live', '{}','Populate comments based on subscriptionshipTo,  SoldTo,  classification  and Latecycle ', 5,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(Comments,PromotionUUID_SP)
print(s_sql)
spark.sql(s_sql)


IncrementalInvoiceCharge = '''CASE WHEN InvoiceCalculationType = "PerCycle" THEN CycleCount * ListPrice ELSE ListPrice END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('IncrementalInvoiceCharge', 'InvoiceAddendumDetails_Live_InvoiceAmt', '{}','Calculate incremental invoice amount based on NoPhoneNumberFlag and ListPrice', 1,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(IncrementalInvoiceCharge,PromotionUUID_SP)
print(s_sql)
spark.sql(s_sql)



Comments = '''CASE WHEN AdjustSubscriptionFlag = 1 THEN "LateCycle_SubscriptionAdjust" ELSE "" END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('Comments', 'AdjustSubscription_TruUp', '{}','Populate comments based on subscription adjust flag for latecycles truup', 1,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(Comments,PromotionUUID_SP)
print(s_sql)
spark.sql(s_sql)



NewSubscriptionStartDate = '''CycleDate'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('NewSubscriptionStartDate', 'AdjustSubscription_TruUp', '{}','Populate new subscriptionstartdate based on latecycle data', 2,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(NewSubscriptionStartDate,PromotionUUID_SP)
print(s_sql)
spark.sql(s_sql)


NewSubscriptionEndDate = '''CASE WHEN ifnull(InvoiceExceptionFlag,"N") = "N" THEN date_add(CycleDate,150) ELSE SubscriptionEndDate END '''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('NewSubscriptionEndDate', 'AdjustSubscription_TruUp', '{}','Populate new subscriptionstartdate based on latecycle data', 3,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(NewSubscriptionEndDate,PromotionUUID_SP)
print(s_sql)
spark.sql(s_sql)


RealignNewSubscriptionFlag = '''CASE WHEN VersionID > 1 AND NKEY_ConsumerSubscriptionUUID IS NULL THEN 1 ELSE 0 END '''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('RealignNewSubscriptionFlag', 'RealignNewSubscription_TruUp', '{}','Populate flag to calculate subscription realignment records', 1,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(RealignNewSubscriptionFlag,PromotionUUID_SP)
print(s_sql)
spark.sql(s_sql)


NewSubscriptionStartDate = '''CycleDate'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('NewSubscriptionStartDate', 'RealignNewSubscription_TruUp', '{}','Populate subscriptionstartdate for realignment records', 2,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(NewSubscriptionStartDate,PromotionUUID_SP)
print(s_sql)
spark.sql(s_sql)


NewSubscriptionEndDate = '''CASE WHEN ifnull(InvoiceExceptionFlag,"N") = "N" THEN date_add(CycleDate,150) ELSE SubscriptionEndDate END '''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('NewSubscriptionEndDate', 'RealignNewSubscription_TruUp', '{}','Populate subscriptionenddate for realignment records', 3,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(NewSubscriptionEndDate,PromotionUUID_SP)
print(s_sql)
spark.sql(s_sql)

Comments = '''"LateCycle_RealignNewSubscription"'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('Comments', 'RealignNewSubscription_TruUp', '{}','popuate comments for realignment records', 4,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(Comments,PromotionUUID_SP)
print(s_sql)
spark.sql(s_sql)


PerPatientFeeFlag = '''CASE WHEN IncrementalInvoiceCharge > 0 THEN 1 WHEN ROW_NUM = 1 AND Prev_SubscriptionInvoiceAmt = 0 THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('PerPatientFeeFlag', 'InvoiceAddendumDetails_TruUp', '{}','calculate PerpatinetFeeFlag for all realignment records', 1,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(PerPatientFeeFlag,PromotionUUID_SP)
print(s_sql)
spark.sql(s_sql)

FollowUpVisitFlag = '''CASE WHEN Prev_SubscriptionInvoiceAmt > 0 THEN 1 WHEN SubscriptionInvoiceAmt = 0 AND ROW_NUM > 1 THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('FollowUpVisitFlag', 'InvoiceAddendumDetails_TruUp', '{}','calculate FollowUpVisitFlag for all realignment records', 2,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(FollowUpVisitFlag,PromotionUUID_SP)
print(s_sql)
spark.sql(s_sql)

PatientClassificationName = '''CASE WHEN PerPatientFeeFlag = 1 THEN "PerPatientFee" WHEN FollowUpVisitFlag = 1 THEN "FollowUpVisit" ELSE "" END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('PatientClassificationName', 'InvoiceAddendumDetails_TruUp', '{}','populate classification based on flags calculated earlierfor all realignment records', 3,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(PatientClassificationName,PromotionUUID_SP)
print(s_sql)
spark.sql(s_sql)

LateCycleFlag = '''CASE WHEN CycleDate <= MaxInvoiceDate THEN 1 ELSE 0 END'''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('LateCycleFlag', 'InvoiceAddendumDetails_TruUp', '{}','Rule to check Late Cycle which needs to adjust consumer subscription', 4,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(LateCycleFlag,PromotionUUID_SP)
print(s_sql)
spark.sql(s_sql)

Comments = '''CASE WHEN LateCycleFlag = 0 THEN "" WHEN Comments_NKEY IS NOT NULL THEN Comments_NKEY WHEN PerPatientFeeFlag = 1 THEN "LateCycle_NewSubscription" WHEN FollowUpVisitFlag = 1 THEN "LateCycle_FollowUp" ELSE "" END '''
s_sql = '''INSERT INTO promotion.STG_RuleEngineConfig (
    RuleName,RuleCategory,RuleCode,RuleComments,RuleOrder,PromotionUUID,DependentRules,
    EffectiveDateTime,TerminationDateTime,CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
    values
    ('Comments', 'InvoiceAddendumDetails_TruUp', '{}','Populate comments for all realignment records', 5,'{}','',
    Current_Date(),'9999-12-31',Current_Date(),'RulesSetup',Current_Date(),'RulesSetup',True)'''.format(Comments,PromotionUUID_SP)
print(s_sql)
spark.sql(s_sql)



In [0]:
%sql
select RuleName,RuleCategory,PromotionUUID,* from promotion.STG_RuleEngineConfig AS source
Order BY PromotionUUID,RuleCategory,RuleName


In [0]:
%sql
-- Update
select  tgt.RuleCode,src.RuleCode,* 
from    promotion.DIM_RuleEngineConfig AS tgt
INNER JOIN promotion.STG_RuleEngineConfig AS src
ON tgt.RuleName = src.RuleName 
   AND tgt.RuleCategory = src.RuleCategory 
   AND tgt.PromotionUUID = src.PromotionUUID
WHERE (tgt.RuleCode <> src.RuleCode 
   OR tgt.RuleComments <> src.RuleComments 
   OR tgt.RuleOrder <> src.RuleOrder 
   OR tgt.Active <> src.Active)

In [0]:
%sql
-- INSERT
select  * 
from    promotion.STG_RuleEngineConfig AS src
LEFT JOIN promotion.DIM_RuleEngineConfig AS tgt
ON tgt.RuleName = src.RuleName 
   AND tgt.RuleCategory = src.RuleCategory 
   AND tgt.PromotionUUID = src.PromotionUUID
WHERE tgt.RuleCode is null

In [0]:
%sql
MERGE INTO promotion.DIM_RuleEngineConfig AS target
USING promotion.STG_RuleEngineConfig AS source
ON target.RuleName = source.RuleName 
   AND target.RuleCategory = source.RuleCategory 
   AND target.PromotionUUID = source.PromotionUUID
WHEN MATCHED AND (target.RuleCode <> source.RuleCode 
                OR target.RuleComments <> source.RuleComments 
                OR target.RuleOrder <> source.RuleOrder 
                OR target.Active <> source.Active) THEN 
    UPDATE SET 
        target.RuleCode = source.RuleCode,
        target.RuleComments = source.RuleComments,
        target.RuleOrder = source.RuleOrder,
        target.DependentRules = source.DependentRules,
        target.EffectiveDateTime = source.EffectiveDateTime,
        target.TerminationDateTime = source.TerminationDateTime,
        target.UpdatedDate = source.UpdatedDate,
        target.UpdatedBy = source.UpdatedBy,
        target.Active = source.Active
WHEN NOT MATCHED THEN
  INSERT (RuleName, RuleCategory, RuleCode, RuleComments, RuleOrder, PromotionUUID, DependentRules, EffectiveDateTime, TerminationDateTime, CreatedDate, CreatedBy, UpdatedDate, UpdatedBy, Active)
    VALUES (source.RuleName, source.RuleCategory, source.RuleCode, source.RuleComments, source.RuleOrder, source.PromotionUUID, source.DependentRules, source.EffectiveDateTime, source.TerminationDateTime, source.CreatedDate, source.CreatedBy, source.UpdatedDate, source.UpdatedBy, source.Active)
-- WHEN NOT MATCHED BY SOURCE THEN
--   DELETE;

In [0]:
%sql
drop table promotion.STG_RuleEngineConfig;

In [0]:
%sql
Select * 
From promotion.DIM_RuleEngineConfig 
where rulename = 'PatientClassificationName'
order by PromotionUUID,RuleCategory,RuleOrder